# Notebook 2 — Create Labels
In this notebook, we create the target label for delivery delay using the actual delivery date and the estimated delivery date.

The label will indicate whether an order was delivered late or on time.


In [1]:
import pandas as pd

ml_table = pd.read_csv("../artifacts/ml_table.csv")

print("ML table shape:", ml_table.shape)
print("Columns:")
print(ml_table.columns.tolist())

ML table shape: (99441, 19)
Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'item_count', 'total_price', 'total_freight', 'unique_products', 'unique_sellers', 'payment_total', 'payment_count']


### Step 1 — Inspect delivery dates

Before creating the label, we check the actual and estimated delivery dates and identify missing delivery dates.


In [2]:
# Check missing delivery dates

print(
    "Missing actual delivery date:",
    ml_table["order_delivered_customer_date"].isna().sum(),
)

print(
    "Missing estimated delivery date:",
    ml_table["order_estimated_delivery_date"].isna().sum(),
)

Missing actual delivery date: 2965
Missing estimated delivery date: 0


### Step 2 — Handle missing delivery dates

Some orders do not have an actual delivery date, so their delivery outcome cannot be determined.

These orders are excluded from label creation because we cannot reliably classify them as late or on-time.


In [3]:
# Check the status of orders with missing delivery dates

missing_delivery = ml_table[ml_table["order_delivered_customer_date"].isna()]

missing_delivery["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

### Step 3 — Create the late delivery label

An order is labeled as late (`1`) if its actual delivery date is later than the estimated delivery date. Otherwise, it is labeled as on-time (`0`).


In [4]:
# Create the late delivery label

ml_table = ml_table.dropna(subset=["order_delivered_customer_date"]).copy()

ml_table["order_delivered_customer_date"] = pd.to_datetime(
    ml_table["order_delivered_customer_date"]
)

ml_table["order_estimated_delivery_date"] = pd.to_datetime(
    ml_table["order_estimated_delivery_date"]
)

# if late = true (1) , no late = false (0)
ml_table["late"] = (
    ml_table["order_delivered_customer_date"]
    > ml_table["order_estimated_delivery_date"]
).astype(int)

print("Labeled rows:", len(ml_table))
print(ml_table["late"].value_counts())

Labeled rows: 96476
late
0    88649
1     7827
Name: count, dtype: int64


### Step 4 — Verify the label

We check a few real orders to confirm that the `late` label matches the actual and estimated delivery dates.


In [7]:
# Check the label on a few real orders

ml_table[
    [
        "order_id",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "late",
    ]
].sample(5, random_state=42)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,late
22500,c58cff333993bb6b7161d7ec1350eef3,2018-04-06 02:32:49,2018-04-18,0
68941,87673b5ccb20de0a91c28cc461105d76,2018-05-23 15:28:28,2018-05-30,0
23988,80b430d0029bb33110ac31d60e87e0b8,2017-12-07 18:43:46,2017-12-27,0
31303,580603672a21252f21fa8a8b4ca85986,2018-04-26 17:44:27,2018-05-08,0
36131,c09f32e7ba9b4a134455b36eeff8fff3,2018-04-13 17:32:07,2018-04-24,0


### Step 5 — Class distribution

We examine the distribution of the target label to determine how many orders were delivered late versus on time.


In [8]:
# Check class distribution

class_counts = ml_table["late"].value_counts()
class_percentages = ml_table["late"].value_counts(normalize=True) * 100

print("Class counts:")
print(class_counts)

print("\nClass percentages:")
print(class_percentages.round(2))

Class counts:
late
0    88649
1     7827
Name: count, dtype: int64

Class percentages:
late
0    91.89
1     8.11
Name: proportion, dtype: float64


### Class imbalance observation

The target is imbalanced: approximately 91.9% of orders are on-time, while 8.1% are late.

Therefore, the `late` class is a minority class and class imbalance should be considered when training and evaluating the model.


### Step 6 — Save the labeled table

The labeled ML table is saved as an artifact and will be used as the input for the next notebook.


In [9]:
# Save the labeled table as an artifact

ml_table.to_csv("../artifacts/labeled_table.csv", index=False)

print("Labeled table saved successfully.")

Labeled table saved successfully.
